# SpaceX Falcon 9 First-Stage Landing Prediction

Portfolio-ready version of the machine-learning stage of the SpaceX Data Science project.

The objective is to predict whether the Falcon 9 first stage will land successfully using the feature-engineered launch dataset created in the preceding project stages.

## Improvements in this version

- Uses standard Python/Jupyter code instead of Pyodide-specific `piplite` and JavaScript fetch calls.
- Splits train/test data **before** fitting preprocessing steps.
- Uses scikit-learn `Pipeline` objects to prevent scaling leakage during cross-validation.
- Compares Logistic Regression, SVM, Decision Tree and KNN under the same evaluation structure.
- Separates cross-validation performance from held-out test performance.
- Adds a final model-comparison figure suitable for the portfolio.

> Note: the dataset is small (90 launches; 18 observations in the held-out test set), so accuracy estimates should be interpreted cautiously.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

%matplotlib inline

## 2. Load modelling data

In [ ]:
DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
FEATURES_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv"

data = pd.read_csv(DATA_URL)
X = pd.read_csv(FEATURES_URL)
y = data["Class"].astype(int)

print(f"Observations: {len(X)}")
print(f"Features: {X.shape[1]}")
print("\nTarget distribution:")
display(y.value_counts().rename(index={0: "No landing", 1: "Successful landing"}))

X.head()

## 3. Train/test split

The original project used a 20% held-out test subset with `random_state=2`.  
The same split size and random state are retained here, but preprocessing is fitted only within the training workflow.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=2,
    stratify=y,
)

print("Training:", X_train.shape)
print("Test:", X_test.shape)

## 4. Model selection

Hyperparameters are selected using stratified 10-fold cross-validation.

Scaling is placed inside the model pipelines, which ensures that each cross-validation fold learns its preprocessing parameters only from that fold's training data.

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=2)

searches = {}

# Logistic Regression
logreg_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000)),
])
searches["Logistic Regression"] = GridSearchCV(
    logreg_pipe,
    {
        "model__C": [0.01, 0.1, 1],
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"],
    },
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

# Support Vector Machine
svm_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", SVC()),
])
searches["SVM"] = GridSearchCV(
    svm_pipe,
    {
        "model__kernel": ["linear", "rbf", "poly", "sigmoid"],
        "model__C": np.logspace(-3, 3, 5),
        "model__gamma": np.logspace(-3, 3, 5),
    },
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

# Decision Tree
tree = DecisionTreeClassifier(random_state=2)
searches["Decision Tree"] = GridSearchCV(
    tree,
    {
        "criterion": ["gini", "entropy"],
        "splitter": ["best", "random"],
        "max_depth": [2, 4, 6, 8, 10, 12],
        "max_features": ["sqrt", "log2", None],
        "min_samples_leaf": [1, 2, 4],
        "min_samples_split": [2, 5, 10],
    },
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

# K-Nearest Neighbours
knn_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", KNeighborsClassifier()),
])
searches["KNN"] = GridSearchCV(
    knn_pipe,
    {
        "model__n_neighbors": list(range(1, 11)),
        "model__algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
        "model__p": [1, 2],
    },
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

for name, search in searches.items():
    print(f"Training {name}...")
    search.fit(X_train, y_train)
    print(f"  Best CV accuracy: {search.best_score_:.3f}")
    print(f"  Best parameters: {search.best_params_}\n")

## 5. Held-out test evaluation

In [ ]:
results = []

for name, search in searches.items():
    y_pred = search.predict(X_test)
    results.append({
        "Model": name,
        "CV Accuracy": search.best_score_,
        "Test Accuracy": accuracy_score(y_test, y_pred),
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("CV Accuracy", ascending=False)
    .reset_index(drop=True)
)

results_df.style.format({
    "CV Accuracy": "{:.3f}",
    "Test Accuracy": "{:.3f}",
})

## 6. Portfolio figure — model comparison

The held-out test set contains only 18 launches. The chart is useful for comparing the models in this exercise, but small differences in accuracy should not be over-interpreted.

In [ ]:
plot_df = results_df.sort_values("Test Accuracy", ascending=True)

plt.figure(figsize=(9, 5.5))
bars = plt.barh(plot_df["Model"], plot_df["Test Accuracy"])

plt.xlim(0, 1)
plt.xlabel("Accuracy en el conjunto de test")
plt.ylabel("")
plt.title("Comparación de modelos — Predicción de aterrizaje Falcon 9")

for bar, score in zip(bars, plot_df["Test Accuracy"]):
    plt.text(
        min(score + 0.015, 0.96),
        bar.get_y() + bar.get_height() / 2,
        f"{score:.1%}",
        va="center",
    )

plt.tight_layout()
plt.show()

## 7. Confusion matrix for the CV-selected model

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_search = searches[best_name]
best_predictions = best_search.predict(X_test)

cm = confusion_matrix(y_test, best_predictions)

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cbar=False,
    xticklabels=["No land", "Land"],
    yticklabels=["No land", "Land"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.show()

print(f"CV-selected model: {best_name}")
print()
print(classification_report(
    y_test,
    best_predictions,
    target_names=["No landing", "Successful landing"],
))

## 8. Summary

This modelling stage compares four supervised classification approaches for Falcon 9 first-stage landing prediction.

### Workflow

**Feature-engineered launch data → Train/test split → Cross-validated hyperparameter tuning → Held-out evaluation → Model comparison**

For portfolio presentation, use the final model-comparison chart together with the interactive launch dashboard from the project.